In [5]:
import numpy as np
import pandas as pd

In [16]:
class Loan:
    def __init__(self, 
                 name:str, 
                 dtype:str, 
                 principal:float, 
                 annual_rate:float, 
                 bidrag_rate:float=0.0, 
                 term_years:float=25, 
                 amort_free_years:float=0.0, 
                 payments_per_year:int=1, 
                 bond_price:float=100
                 ):

        self.name = name
        self.dtype = dtype
        self.principal = principal
        self.annual_rate = annual_rate
        self.bidrag_rate = bidrag_rate
        self.term_years = term_years
        self.amort_free_years = amort_free_years
        self.payments_per_year = payments_per_year
        self.bond_price = bond_price
        self.effective_interest_rate = (annual_rate + bidrag_rate)/payments_per_year

        self.number_of_payments = round(
            term_years * payments_per_year
        )
        self.interest_only_payments = round(
            amort_free_years * payments_per_year
        )

    @staticmethod
    def compute_total_payment(amount, interest, nterms):
        return amount * (interest*(1 + interest)**nterms)/((1 + interest)**nterms - 1)



    def compute_amort_payment(self, outstanding):
        payback_time = self.term_years - self.amort_free_years
        TNP = self.compute_total_payment(outstanding, self.effective_interest_rate, payback_time)
        return TNP - (outstanding * self.effective_interest_rate)



    def calculate_schedule(self):
        balance = self.principal
        payment = self.compute_amort_payment(balance)
        interest_rate = self.effective_interest_rate

        schedule = []

        for period in range(1, self.number_of_payments + 1):

            beginning_balance = balance
            interest = beginning_balance * interest_rate

            # Interest-only period
            if period <= self.interest_only_payments:

                principal_payment = 0
                total_payment = interest

            else:

                principal_payment = (
                    payment
                    - interest
                )

                principal_payment = min(
                    principal_payment,
                    balance
                )

                total_payment = (
                    interest
                    + principal_payment
                )

            balance -= principal_payment

            if balance < 0:
                balance = 0

            # Calculate market value using bond price
            if self.bond_price is not None:
                market_value = (
                    balance
                    * self.bond_price
                    / 100
                )
            else:
                market_value = None

            schedule.append({
                "Loan": self.name,
                "Period": period,
                "Beginning Balance": beginning_balance,
                "Interest": interest,
                "Principal Payment": principal_payment,
                "Total Payment": total_payment,
                "Remaining Balance": balance,
                "Bond Price": self.bond_price,
                "Market Value": market_value
            })

        return pd.DataFrame(schedule)

    # def annual_schedule(self):
    #     """
    #     Return the loan development summarized by year.
    #     """

    #     schedule = self.calculate_schedule()

    #     schedule["Year"] = (
    #         (schedule["Period"] - 1)
    #         // self.payments_per_year
    #         + 1
    #     )

    #     annual = schedule.groupby("Year").agg({
    #         "Interest": "sum",
    #         "Contribution": "sum",
    #         "Principal Payment": "sum",
    #         "Total Payment": "sum",
    #         "Remaining Balance": "last",
    #         "Market Value": "last"
    #     }).reset_index()

    #     return annual

    # def remaining_balance(self, year):
    #     """
    #     Return the remaining debt after a given number of years.
    #     """

    #     schedule = self.annual_schedule()

    #     if year == 0:
    #         return self.principal

    #     row = schedule[schedule["Year"] == year]

    #     if row.empty:
    #         raise ValueError(
    #             "Year is outside the loan term."
    #         )

    #     return row.iloc[0]["Remaining Balance"]

    # def market_value(self, year=0):
    #     """
    #     Calculate the market value of the remaining debt.
    #     """

    #     if self.bond_price is None:
    #         raise ValueError(
    #             "Bond price has not been specified."
    #         )

    #     if year == 0:
    #         balance = self.principal
    #     else:
    #         balance = self.remaining_balance(year)

    #     return balance * self.bond_price / 100

    # def total_interest(self):
    #     """Return total interest paid over the lifetime of the loan."""

    #     schedule = self.calculate_schedule()

    #     return schedule["Interest"].sum()

    # def total_contribution(self):
    #     """Return total contribution paid over the lifetime."""

    #     schedule = self.calculate_schedule()

    #     return schedule["Contribution"].sum()

    # def total_cost(self):
    #     """Return total payments over the lifetime."""

    #     schedule = self.calculate_schedule()

    #     return schedule["Total Payment"].sum()

    # def summary(self):
    #     """Print a simple summary of the loan."""

    #     print(f"Loan: {self.name}")
    #     print(f"Principal:          {self.principal:,.2f}")
    #     print(f"Interest rate:      {self.interest_rate * 100:.4f}%")
    #     print(f"Contribution rate:  {self.contribution_rate * 100:.4f}%")
    #     print(f"Term:               {self.term_years} years")
    #     print(f"Interest-only:      {self.interest_only_years} years")

    #     if self.bond_price is not None:
    #         print(f"Bond price:         {self.bond_price:.3f}")

    #     payment = self.calculate_payment()

    #     if payment is not None:
    #         print(f"Regular payment:    {payment:,.2f}")

    #     print(f"Total interest:     {self.total_interest():,.2f}")
    #     print(f"Total contribution: {self.total_contribution():,.2f}")
    #     print(f"Total cost:         {self.total_cost():,.2f}")


loan_16 = Loan(
    name="Loan 16",
    dtype="Cash Loan",
    principal=12_274_139.55,
    annual_rate=0.010612,
    bidrag_rate=0.004208,       # enter actual rate
    term_years=24,
    payments_per_year=4,
    amort_free_years=0,
    bond_price=77.43
)

loan_16.calculate_schedule()
# loan_16.compute_total_payment(loan_16.principal, 0.010612 + 0.004208, 25)




,Loan,Period,Beginning Balance,Interest,Principal Payment,Total Payment,Remaining Balance,Bond Price,Market Value
0,Loan 16,1,1.227414e+07,45475.687033,444492.084883,489967.771916,1.182965e+07,77.43,9.159696e+06
1,Loan 16,2,1.182965e+07,43828.843858,446138.928058,489967.771916,1.138351e+07,77.43,8.814251e+06
2,Loan 16,3,1.138351e+07,42175.899130,447791.872786,489967.771916,1.093572e+07,77.43,8.467525e+06
3,Loan 16,4,1.093572e+07,40516.830241,449450.941675,489967.771916,1.048627e+07,77.43,8.119516e+06
4,Loan 16,5,1.048627e+07,38851.614502,451116.157414,489967.771916,1.003515e+07,77.43,7.770216e+06
...,...,...,...,...,...,...,...,...,...
91,Loan 16,92,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,77.43,0.000000e+00
92,Loan 16,93,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,77.43,0.000000e+00
93,Loan 16,94,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,77.43,0.000000e+00
94,Loan 16,95,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,77.43,0.000000e+00


In [8]:
loan_15 = Loan(
    name="Loan 15",
    principal=5_493_000,
    interest_rate=0.0100,
    contribution_rate=0.003889,
    term_years=24.25,
    interest_only_years=24.25,
    bond_price=80.938
)

loan_16 = Loan(
    name="Loan 16",
    principal=12_274_139.55,
    interest_rate=0.015,
    contribution_rate=0.0,       # enter actual rate
    term_years=24,
    payments_per_year=4,
    interest_only_years=0,
    bond_price=77.43
)

loan_17 = Loan(
    name="Loan 17",
    principal=8_795_398.91,
    interest_rate=0.02116,           # enter actual rate
    contribution_rate=0.0,       # enter actual rate
    term_years=24.5,
    payments_per_year=4,
    interest_only_years=0,
    bond_price=80.80
)

# loans = [loan_15, loan_16, loan_17]

# for loan in loans:
#     loan.summary()
#     print()

loan_16.calculate_schedule()[0:4].sum()
# loan_17.calculate_schedule()[0:4].sum()




TypeError: Loan.__init__() got an unexpected keyword argument 'interest_rate'

In [ ]:
comparison = pd.DataFrame({
    loan.name: {
        "Initial balance": loan.principal,
        "Interest rate": loan.interest_rate,
        "Contribution rate": loan.contribution_rate,
        "Total interest": loan.total_interest(),
        "Total contribution": loan.total_contribution(),
        "Total cost": loan.total_cost()
    }
    for loan in loans
}).T

print(comparison)

         Initial balance  Interest rate  Contribution rate  Total interest  \
Loan 15       5493000.00       0.010000           0.003889    1.318320e+06   
Loan 16      12274139.55       0.010612           0.000000    1.764706e+06   
Loan 17       8795398.91       0.000000           0.000000    0.000000e+00   

         Total contribution    Total cost  
Loan 15          512694.648  1.831015e+06  
Loan 16               0.000  1.403885e+07  
Loan 17               0.000  8.795399e+06  


In [ ]:
loan_16.calculate_schedule()

,Loan,Period,Beginning Balance,Interest,Contribution,Principal Payment,Total Payment,Remaining Balance,Bond Price,Market Value
0,Loan 16,1,1.227414e+07,130253.168905,0.0,431300.666413,561553.835318,1.184284e+07,77.43,9.169910e+06
1,Loan 16,2,1.184284e+07,125676.206233,0.0,435877.629085,561553.835318,1.140696e+07,77.43,8.832410e+06
2,Loan 16,3,1.140696e+07,121050.672833,0.0,440503.162485,561553.835318,1.096646e+07,77.43,8.491329e+06
3,Loan 16,4,1.096646e+07,116376.053272,0.0,445177.782046,561553.835318,1.052128e+07,77.43,8.146627e+06
4,Loan 16,5,1.052128e+07,111651.826649,0.0,449902.008669,561553.835318,1.007138e+07,77.43,7.798268e+06
5,Loan 16,6,1.007138e+07,106877.466533,0.0,454676.368785,561553.835318,9.616702e+06,77.43,7.446212e+06
6,Loan 16,7,9.616702e+06,102052.440908,0.0,459501.394410,561553.835318,9.157201e+06,77.43,7.090420e+06
7,Loan 16,8,9.157201e+06,97176.212110,0.0,464377.623208,561553.835318,8.692823e+06,77.43,6.730853e+06
8,Loan 16,9,8.692823e+06,92248.236773,0.0,469305.598545,561553.835318,8.223517e+06,77.43,6.367469e+06
9,Loan 16,10,8.223517e+06,87267.965761,0.0,474285.869557,561553.835318,7.749231e+06,77.43,6.000230e+06


In [ ]:
r = 0.01
R = 76291.64
Ob = 0.809330
TOT = 5493000



(R - TOT*r)/TOT





0.0038888840342253777

In [ ]:
TOT17 = 8992965.83
tot17 = 8795398.91
r17 = 0.016868
R17 = 186130
Ob17 = 0.808

TOT17/Ob17*r17



187739.29160945545